# pyopenssl Package Testing

This notebook tests the pyopenssl package, which provides Python bindings for OpenSSL library functionality including SSL/TLS and cryptographic operations.

## Example 1: SSL Context and Certificate Basics

In [ ]:
import OpenSSL
from OpenSSL import SSL, crypto
import socket
import datetime

print("Testing pyOpenSSL SSL context and certificates:")

# Create SSL context
try:
    context = SSL.Context(SSL.TLSv1_2_METHOD)
    print("✅ SSL context created successfully")
    print(f"Context method: {context}")
    
    # Test context options
    context.set_options(SSL.OP_NO_SSLv2 | SSL.OP_NO_SSLv3)
    print("✅ SSL options set successfully")
    
    # Create a simple self-signed certificate for testing
    key = crypto.PKey()
    key.generate_key(crypto.TYPE_RSA, 2048)
    print("✅ RSA key pair generated")
    
    cert = crypto.X509()
    cert.get_subject().C = "US"
    cert.get_subject().ST = "State"
    cert.get_subject().L = "City"
    cert.get_subject().O = "Organization"
    cert.get_subject().CN = "localhost"
    
    cert.set_serial_number(1000)
    cert.gmtime_adj_notBefore(0)
    cert.gmtime_adj_notAfter(365*24*60*60)  # Valid for 1 year
    cert.set_issuer(cert.get_subject())
    cert.set_pubkey(key)
    cert.sign(key, 'sha256')
    
    print("✅ Self-signed certificate created")
    print(f"Certificate subject: {cert.get_subject()}")
    print(f"Certificate serial: {cert.get_serial_number()}")
    
except Exception as e:
    print(f"❌ SSL setup error: {e}")
    raise AssertionError(
        f"pyopenssl example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Certificate Information and Validation

In [ ]:
import OpenSSL
from OpenSSL import crypto
import datetime

print("Testing certificate information extraction:")

try:
    # Create a test certificate
    key = crypto.PKey()
    key.generate_key(crypto.TYPE_RSA, 1024)
    
    cert = crypto.X509()
    subject = cert.get_subject()
    subject.countryName = "US"
    subject.stateOrProvinceName = "California"  
    subject.localityName = "San Francisco"
    subject.organizationName = "Test Company"
    subject.organizationalUnitName = "IT Department"
    subject.commonName = "test.example.com"
    subject.emailAddress = "admin@example.com"
    
    cert.set_serial_number(12345)
    cert.gmtime_adj_notBefore(0)
    cert.gmtime_adj_notAfter(365*24*60*60)  # 1 year validity
    cert.set_issuer(cert.get_subject())
    cert.set_pubkey(key)
    cert.sign(key, 'sha256')
    
    print("Certificate Details:")
    print(f"  Subject: {cert.get_subject()}")
    print(f"  Issuer: {cert.get_issuer()}")
    print(f"  Serial Number: {cert.get_serial_number()}")
    print(f"  Version: {cert.get_version()}")
    
    # Get validity period
    not_before = cert.get_notBefore().decode('ascii')
    not_after = cert.get_notAfter().decode('ascii')
    print(f"  Valid From: {not_before}")
    print(f"  Valid Until: {not_after}")
    
    # Check if certificate has expired
    has_expired = cert.has_expired()
    print(f"  Has Expired: {has_expired}")
    
    # Get public key info
    pub_key = cert.get_pubkey()
    print(f"  Public Key Type: {pub_key.type()}")
    print(f"  Public Key Bits: {pub_key.bits()}")
    
    # Certificate extensions (if any)
    ext_count = cert.get_extension_count()
    print(f"  Extensions Count: {ext_count}")
    
    print("✅ Certificate information extracted successfully")
    
except Exception as e:
    print(f"❌ Certificate processing error: {e}")
    raise AssertionError(
        f"pyopenssl example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Cryptographic Operations

In [ ]:
import OpenSSL
from OpenSSL import crypto
import base64

# API drift: `crypto.sign` / `crypto.verify` were DEPRECATED and are now REMOVED from pyOpenSSL —
# `dir(crypto)` no longer carries either. pyOpenSSL's own guidance is to use `cryptography` directly
# for raw sign/verify, which is what these imports are for. `PKey.to_cryptography_key()` is the
# bridge between the two object models.
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.exceptions import InvalidSignature

print("Testing cryptographic operations:")

try:
    # Generate RSA key pair
    key = crypto.PKey()
    key.generate_key(crypto.TYPE_RSA, 2048)
    print("✅ RSA key pair generated (2048 bits)")
    
    # Test message to sign/verify
    message = b"Hello, OpenSSL world!"
    print(f"Original message: {message.decode()}")
    
    # Sign the message
    signature = key.to_cryptography_key().sign(message, padding.PKCS1v15(), hashes.SHA256())
    print(f"✅ Message signed with SHA-256")
    print(f"Signature length: {len(signature)} bytes")
    print(f"Signature (base64): {base64.b64encode(signature)[:50].decode()}...")
    
    # Verify the signature
    cert = crypto.X509()
    cert.set_pubkey(key)
    
    public_key = cert.get_pubkey().to_cryptography_key()
    
    public_key.verify(signature, message, padding.PKCS1v15(), hashes.SHA256())
    print("✅ Signature verification successful")
    
    # Test with tampered message
    tampered_message = b"Hello, tampered world!"
    # The REJECTION is the assertion here: a tampered message that verifies is the failure, so it
    # raises rather than printing. `cryptography` signals this as `InvalidSignature`, not `crypto.Error`.
    try:
        public_key.verify(signature, tampered_message, padding.PKCS1v15(), hashes.SHA256())
    except InvalidSignature:
        print("✅ Tampered message correctly rejected")
    else:
        raise AssertionError(
            "a tampered message VERIFIED against the original signature — signature checking is broken"
        )
    
    # Export keys in different formats
    private_pem = crypto.dump_privatekey(crypto.FILETYPE_PEM, key)
    public_pem = crypto.dump_publickey(crypto.FILETYPE_PEM, key)
    
    print(f"\nPrivate key (PEM, first 100 chars):")
    print(private_pem[:100].decode())
    print(f"\nPublic key (PEM, first 100 chars):")
    print(public_pem[:100].decode())
    
except Exception as e:
    print(f"❌ Cryptographic operation error: {e}")
    raise AssertionError(
        f"pyopenssl example (cell 6) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Certificate Store and Chain Validation

In [ ]:
import OpenSSL
from OpenSSL import crypto
import tempfile
import os
import datetime

# API drift: `X509.add_extensions` and `crypto.X509Extension` are both REMOVED — pyOpenSSL's X509 is
# read-only for extensions now (`get_extension_count` is all that survives). Certificates carrying
# extensions are built with `cryptography`'s CertificateBuilder and bridged back with
# `X509.from_cryptography()`, which is the path pyOpenSSL itself documents.
from cryptography import x509 as cryptography_x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes as cryptography_hashes

print("Testing certificate store and chain validation:")

try:
    # Create a CA certificate
    ca_key = crypto.PKey()
    ca_key.generate_key(crypto.TYPE_RSA, 2048)
    
    # The extensions below are LOAD-BEARING: without `basicConstraints CA:TRUE` the store refuses
    # this certificate as an issuer, so the chain validation further down would fail. That is why
    # this is rebuilt on the cryptography builder rather than simply dropping the extensions.
    ca_private_key = ca_key.to_cryptography_key()
    ca_name = cryptography_x509.Name([
        cryptography_x509.NameAttribute(NameOID.COUNTRY_NAME, "US"),
        cryptography_x509.NameAttribute(NameOID.ORGANIZATION_NAME, "Test CA"),
        cryptography_x509.NameAttribute(NameOID.COMMON_NAME, "Test Root CA"),
    ])
    _now = datetime.datetime.now(datetime.timezone.utc)
    ca_cert = crypto.X509.from_cryptography(
        cryptography_x509.CertificateBuilder()
        .subject_name(ca_name)
        .issuer_name(ca_name)  # Self-signed
        .public_key(ca_private_key.public_key())
        .serial_number(1)
        .not_valid_before(_now)
        .not_valid_after(_now + datetime.timedelta(days=10 * 365))  # 10 years
        .add_extension(
            cryptography_x509.BasicConstraints(ca=True, path_length=None), critical=True
        )
        .add_extension(
            cryptography_x509.KeyUsage(
                digital_signature=False, content_commitment=False, key_encipherment=False,
                data_encipherment=False, key_agreement=False, key_cert_sign=True,
                crl_sign=True, encipher_only=False, decipher_only=False,
            ),
            critical=True,
        )
        .sign(ca_private_key, cryptography_hashes.SHA256())
    )
    print(f"✅ CA certificate created ({ca_cert.get_extension_count()} extensions)")
    
    # Create an end-entity certificate signed by CA
    ee_key = crypto.PKey() 
    ee_key.generate_key(crypto.TYPE_RSA, 2048)
    
    ee_cert = crypto.X509()
    ee_subject = ee_cert.get_subject()
    ee_subject.countryName = "US"
    ee_subject.organizationName = "Test Organization"
    ee_subject.commonName = "test.example.com"
    
    ee_cert.set_serial_number(2)
    ee_cert.gmtime_adj_notBefore(0)
    ee_cert.gmtime_adj_notAfter(365*24*60*60)  # 1 year
    ee_cert.set_issuer(ca_cert.get_subject())  # Signed by CA
    ee_cert.set_pubkey(ee_key)
    ee_cert.sign(ca_key, 'sha256')  # Sign with CA key
    
    print("✅ End-entity certificate created")
    
    # Create certificate store and add CA
    store = crypto.X509Store()
    store.add_cert(ca_cert)
    print("✅ Certificate store created with CA")
    
    # Verify end-entity certificate against store
    store_context = crypto.X509StoreContext(store, ee_cert)
    
    try:
        store_context.verify_certificate()
        print("✅ Certificate chain validation successful")
    except crypto.X509StoreContextError as e:
        print(f"❌ Certificate validation failed: {e}")
        raise  # the CA-signed chain must verify against a store holding that CA
    
    # Test with invalid certificate (self-signed)
    invalid_cert = crypto.X509()
    invalid_cert.set_subject(ee_cert.get_subject())
    invalid_cert.set_serial_number(999)
    invalid_cert.gmtime_adj_notBefore(0)
    invalid_cert.gmtime_adj_notAfter(365*24*60*60)
    invalid_cert.set_issuer(invalid_cert.get_subject())  # Self-signed
    invalid_cert.set_pubkey(ee_key)
    invalid_cert.sign(ee_key, 'sha256')  # Self-signed
    
    invalid_store_context = crypto.X509StoreContext(store, invalid_cert)
    try:
        invalid_store_context.verify_certificate()
    except crypto.X509StoreContextError as e:
        # e.errors = [OpenSSL verify code, depth, message]; 18 = X509_V_ERR_DEPTH_ZERO_SELF_SIGNED_CERT
        assert e.errors[0] == 18, f"rejected for an unexpected reason: {e.errors}"
        print(f"✅ Invalid certificate correctly rejected ({e})")
    else:
        raise AssertionError("a self-signed certificate verified against a store that does not hold it")
        
except Exception as e:
    print(f"❌ Certificate store error: {e}")
    raise AssertionError(
        f"pyopenssl example (cell 8) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: SSL Configuration Testing

In [ ]:
import OpenSSL
from OpenSSL import SSL
import socket

print("Testing SSL configuration and capabilities:")

try:
    # Test different SSL methods
    ssl_methods = [
        ("TLSv1_2_METHOD", SSL.TLSv1_2_METHOD),
        ("SSLv23_METHOD", SSL.SSLv23_METHOD),
    ]
    
    for method_name, method in ssl_methods:
        context = SSL.Context(method)
        assert isinstance(context, SSL.Context), context
        print(f"✅ {method_name} context created successfully")

    # Test SSL options: set_options returns the context's new option mask
    context = SSL.Context(SSL.TLSv1_2_METHOD)
    ssl_options = [
        ("OP_NO_SSLv2", SSL.OP_NO_SSLv2),
        ("OP_NO_SSLv3", SSL.OP_NO_SSLv3),
        ("OP_NO_COMPRESSION", SSL.OP_NO_COMPRESSION),
    ]

    for option_name, option in ssl_options:
        mask = context.set_options(option)
        assert (mask & option) == option, (option_name, hex(mask))
        print(f"✅ {option_name} set successfully")
    # OP_NO_SSLv2 is 0 since OpenSSL 1.1 removed SSLv2 (measured: 0x0), so the two real bits are these
    assert (mask & (SSL.OP_NO_SSLv3 | SSL.OP_NO_COMPRESSION)) == SSL.OP_NO_SSLv3 | SSL.OP_NO_COMPRESSION, hex(mask)

    # Test cipher configuration (an unknown cipher string raises SSL.Error: no cipher match)
    context.set_cipher_list(b"ECDHE+AESGCM:ECDHE+CHACHA20:DHE+AESGCM:DHE+CHACHA20:!aNULL:!MD5:!DSS")
    print("✅ Cipher list configured")

    # Test verification configuration
    context.set_verify(SSL.VERIFY_PEER, lambda conn, cert, errno, depth, ok: ok)
    assert context.get_verify_mode() == SSL.VERIFY_PEER, context.get_verify_mode()
    print("✅ Verification callback set")
    
    # Test certificate and key loading
    try:
        # Generate test key and certificate. 2048 bits: OpenSSL's default security level (2) rejects a
        # 1024-bit RSA key in an SSL context ("ee key too small").
        key = OpenSSL.crypto.PKey()
        key.generate_key(OpenSSL.crypto.TYPE_RSA, 2048)
        
        cert = OpenSSL.crypto.X509()
        cert.get_subject().CN = "test"
        cert.set_serial_number(1)
        cert.gmtime_adj_notBefore(0)
        cert.gmtime_adj_notAfter(365*24*60*60)
        cert.set_issuer(cert.get_subject())
        cert.set_pubkey(key)
        cert.sign(key, 'sha256')
        
        # Load into context
        context.use_privatekey(key)
        context.use_certificate(cert)
        context.check_privatekey()
        
        assert key.bits() == 2048 and cert.get_subject().CN == "test"
        print("✅ Certificate and private key loaded and verified")
        
    except Exception as e:
        print(f"❌ Certificate/key loading failed: {e}")
        raise

    # The cipher list took effect. Checked LAST: creating a Connection freezes the context (pyOpenSSL 26
    # refuses any later mutation), and the key/cert loading above still mutates it.
    enabled = SSL.Connection(context, None).get_cipher_list()
    tls12 = [c for c in enabled if not c.startswith("TLS_")]  # TLS 1.3 suites are configured separately
    assert tls12 and all(c.startswith(("ECDHE-", "DHE-")) and ("GCM" in c or "CHACHA20" in c) for c in tls12), enabled
    print(f"✅ {len(tls12)} TLS 1.2 cipher suites enabled, all ECDHE/DHE with AEAD")
    
    print("\n✅ SSL configuration testing completed")
    
except Exception as e:
    print(f"❌ SSL configuration error: {e}")
    raise AssertionError(
        f"pyopenssl example (cell 10) failed: {type(e).__name__}: {e}"
    ) from e